# Professional Data Cleaning

**Objective:** Audit quality, justify missing-value and outlier treatment, standardise fields and export a clean CSV.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np
df=pd.read_csv('messy_customers_synthetic.csv')
def report(x):
    return {'rows':len(x),'null_cells':int(x.isna().sum().sum()),'duplicate_rows':int(x.duplicated().sum()),'dtypes':x.dtypes.astype(str).to_dict()}
before=report(df); print('Before:',before); display(df.isna().sum()); display(df.describe(include='all'))
df=df.drop_duplicates().copy()
df['Name']=df.Name.fillna('Unknown').str.strip().str.title()
df['Gender']=df.Gender.astype('string').str.strip().str.lower().replace({'f':'female','m':'male'})
df['Gender']=df.Gender.replace({'female':'Female','male':'Male','other':'Other'}).fillna('Unknown')
df['JoinDate']=pd.to_datetime(df.JoinDate,errors='coerce',dayfirst=True)
df['Age']=pd.to_numeric(df.Age,errors='coerce'); df.loc[~df.Age.between(0,100),'Age']=np.nan
df['Spend']=pd.to_numeric(df.Spend.replace('N/A',np.nan),errors='coerce')
df['Age']=df.Age.fillna(df.Age.median()); df['Spend']=df.Spend.fillna(df.Spend.median())
df['JoinDate']=df.JoinDate.fillna(df.JoinDate.median())
df['CustomerID']=df.CustomerID.astype(str)
# IQR flag; retain plausible high spend but cap extreme values to upper fence
q1,q3=df.Spend.quantile([.25,.75]); iqr=q3-q1; upper=q3+1.5*iqr
df['Spend']=df.Spend.clip(upper=upper)
after=report(df); print('After:',after)
display(pd.DataFrame({'Before':[before['rows'],before['null_cells'],before['duplicate_rows']],
                      'After':[after['rows'],after['null_cells'],after['duplicate_rows']]},
                     index=['Rows','Null cells','Duplicate rows']))
df.to_csv('cleaned_customers.csv',index=False)
print('Saved cleaned_customers.csv. Decisions: median for numeric fields, Unknown for categorical name/gender, invalid ages treated as missing, dates parsed with coercion, spend capped using IQR.')
